# Pyr 3D Plotly Viewer

This notebook is the polished Plotly viewer derived from the exploratory spatial-visualization work in Notebook 06. It supports two visualization modes: a 3D centroid view of the screened root population without meshes, and an optional 3D view with selected local decimated meshes overlaid.

The annotation CSV is used only as context for roots already annotated as CA3 cells; unlabeled screened roots are not interpreted as glia or any other cell class here.


## Imports and Paths

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go


def find_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers and data."
    )


project_root = find_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from path_behavior import format_path, print_path

# Set to True to show full absolute paths in user-facing notebook output.
show_full_path = False

In [2]:
# ------------------------------------------------------------
# USER SETTINGS: GENERAL
# ------------------------------------------------------------
nuclei_data_dir = project_root / "data" / "nuclei"
mesh_data_dir = project_root / "data" / "meshes"
decimated_mesh_dir = mesh_data_dir / "dec"
synapse_table_dir = project_root / "data" / "synapse_tables"
plotly_output_dir = project_root / "outputs" / "plotly"

screened_root_parquet_path = nuclei_data_dir / "c3_nuclei_root_features_screened_mat195.parquet"
nuclei_parquet_path = nuclei_data_dir / "c3_nuclei_v1_mat195.parquet"
annotation_csv_path = project_root / "data" / "mouse_hippocampus_ca3_cell_annotations_export.csv"

materialization_version = 195
voxel_resolution_nm = {'x': 18, 'y': 18, 'z': 45}
decimated_mesh_glob = f"mesh_*_mat{materialization_version}_dec*.ply"

# ------------------------------------------------------------
# USER SETTINGS: PLOTLY DISPLAY
# ------------------------------------------------------------
plotly_renderer = "browser"  # "browser" for external window; None for notebook default/inline

print_path("screened root Parquet", screened_root_parquet_path, project_root, show_full_path)
print_path("nuclei Parquet", nuclei_parquet_path, project_root, show_full_path)
print_path("annotation CSV", annotation_csv_path, project_root, show_full_path)
print_path("decimated mesh directory", decimated_mesh_dir, project_root, show_full_path)
print_path("synapse table directory", synapse_table_dir, project_root, show_full_path)
print_path("Plotly output directory", plotly_output_dir, project_root, show_full_path)

screened root Parquet: data\nuclei\c3_nuclei_root_features_screened_mat195.parquet
nuclei Parquet: data\nuclei\c3_nuclei_v1_mat195.parquet
annotation CSV: data\mouse_hippocampus_ca3_cell_annotations_export.csv
decimated mesh directory: data\meshes\dec
synapse table directory: data\synapse_tables
Plotly output directory: outputs\plotly


## Load Screened Roots and Annotations

In [3]:
screened_root_df = pd.read_parquet(screened_root_parquet_path)
annotation_df = pd.read_csv(annotation_csv_path)

screened_root_df = screened_root_df.copy()
annotation_df = annotation_df.copy()

screened_root_df['pt_root_id'] = pd.to_numeric(
    screened_root_df['pt_root_id'],
    errors='raise',
).astype('Int64')

annotation_df['annotation_root_id'] = pd.to_numeric(
    annotation_df['Cell ID'],
    errors='raise',
).astype('Int64')

screened_root_ids = set(screened_root_df['pt_root_id'].dropna().astype(int).tolist())
annotation_root_ids = set(annotation_df['annotation_root_id'].dropna().astype(int).tolist())

screened_root_df['population_label'] = screened_root_df['pt_root_id'].isin(annotation_root_ids).map({
    True: 'annotated_neuron',
    False: 'unlabeled',
})

print(f"screened_root_df shape: {screened_root_df.shape}")
print(f"annotation_df shape: {annotation_df.shape}")
print(f"screened root IDs: {len(screened_root_ids)}")
print(f"annotation root IDs: {len(annotation_root_ids)}")
print(f"overlap root IDs: {len(screened_root_ids & annotation_root_ids)}")
print('population counts:')
display(
    screened_root_df['population_label']
    .value_counts()
    .rename_axis('population_label')
    .reset_index(name='root_count')
)

screened_root_df shape: (13860, 21)
annotation_df shape: (2764, 11)
screened root IDs: 13860
annotation root IDs: 2764
overlap root IDs: 2061
population counts:


,population_label,root_count
0,unlabeled,11799
1,annotated_neuron,2061


## Join Annotation Fields

In [4]:
annotation_fields_to_inspect = ['type', 'subtypes', 'outputs', 'inputs']
available_annotation_fields = [
    field for field in annotation_fields_to_inspect
    if field in annotation_df.columns
]

annotation_join_columns = ['annotation_root_id'] + available_annotation_fields
annotation_fields_for_join_df = annotation_df[annotation_join_columns].copy()
annotation_fields_for_join_df = annotation_fields_for_join_df.rename(columns={
    'type': 'annotation_type',
    'subtypes': 'annotation_subtypes',
    'outputs': 'annotation_outputs',
    'inputs': 'annotation_inputs',
})

screened_root_with_annotation_df = screened_root_df.merge(
    annotation_fields_for_join_df,
    left_on='pt_root_id',
    right_on='annotation_root_id',
    how='left',
    validate='one_to_one',
)

if 'annotation_root_id' in screened_root_with_annotation_df.columns:
    screened_root_with_annotation_df = screened_root_with_annotation_df.drop(columns=['annotation_root_id'])

feature_source_df = screened_root_with_annotation_df.copy()

print('joined dataframe shape:')
print(feature_source_df.shape)
print('available selected nucleus-derived feature columns:')
print([
    column for column in [
        'nuclei_rows',
        'unique_supervoxels',
        'nucleus_volume_sum',
        'nucleus_volume_max',
        'nucleus_volume_median',
        'x_range',
        'y_range',
        'z_range',
    ]
    if column in feature_source_df.columns
])
print('population labels:')
print(feature_source_df['population_label'].value_counts().to_dict())

joined dataframe shape:
(13860, 25)
available selected nucleus-derived feature columns:
['nuclei_rows', 'unique_supervoxels', 'nucleus_volume_sum', 'nucleus_volume_max', 'nucleus_volume_median', 'x_range', 'y_range', 'z_range']
population labels:
{'unlabeled': 11799, 'annotated_neuron': 2061}


## Mesh File Manifest

The centroid-only viewer does not require mesh files. Mesh overlays require local decimated PLY meshes in `data/meshes/dec`, with filenames approximately matching `mesh_<root_id>_mat195_dec95.ply`. These are generated local artifacts and are not necessarily included in the GitHub repository.

Requested meshes that are missing are reported and skipped, so the base centroid viewer can still render. The mesh-enabled section also requires `pyvista` to load the local PLY files.


In [5]:
def refresh_decimated_mesh_manifest():
    decimated_mesh_paths = sorted(decimated_mesh_dir.glob(decimated_mesh_glob))

    decimated_mesh_manifest_df = pd.DataFrame(
        [
            {
                'mesh_path': mesh_path,
                'mesh_filename': mesh_path.name,
                'pt_root_id': int(mesh_path.name.split('_')[1]),
            }
            for mesh_path in decimated_mesh_paths
        ]
    )

    return decimated_mesh_paths, decimated_mesh_manifest_df

decimated_mesh_paths, decimated_mesh_manifest_df = refresh_decimated_mesh_manifest()

print(f'decimated mesh files found: {len(decimated_mesh_manifest_df)}')
decimated_mesh_manifest_display_df = decimated_mesh_manifest_df.head().copy()
if 'mesh_path' in decimated_mesh_manifest_display_df.columns:
    decimated_mesh_manifest_display_df['mesh_path'] = decimated_mesh_manifest_display_df['mesh_path'].map(
        lambda path: format_path(path, project_root, show_full_path)
    )
display(decimated_mesh_manifest_display_df)

decimated mesh files found: 3357


,mesh_path,mesh_filename,pt_root_id
0,data\meshes\dec\mesh_648518346341734247_mat195...,mesh_648518346341734247_mat195_dec95.ply,648518346341734247
1,data\meshes\dec\mesh_648518346342322415_mat195...,mesh_648518346342322415_mat195_dec95.ply,648518346342322415
2,data\meshes\dec\mesh_648518346342407091_mat195...,mesh_648518346342407091_mat195_dec95.ply,648518346342407091
3,data\meshes\dec\mesh_648518346342407859_mat195...,mesh_648518346342407859_mat195_dec95.ply,648518346342407859
4,data\meshes\dec\mesh_648518346342409907_mat195...,mesh_648518346342409907_mat195_dec95.ply,648518346342409907


## Prepare 3D Centroids

Prepare one centroid per screened root for the 3D viewer. If centroid columns are already present they are reused; otherwise the local nuclei table is loaded and volume-weighted centroids are computed from nucleus positions.


In [6]:
centroid_candidate_columns = [
    column for column in screened_root_with_annotation_df.columns
    if any(token in column.lower() for token in ['centroid', 'center'])
]

print('screened joined dataframe shape:')
print(screened_root_with_annotation_df.shape)
print('centroid-like columns already present:')
print(centroid_candidate_columns)
print('available position extent columns:')
print([column for column in ['x_min', 'x_max', 'y_min', 'y_max', 'z_min', 'z_max'] if column in screened_root_with_annotation_df.columns])

screened joined dataframe shape:
(13860, 25)
centroid-like columns already present:
[]
available position extent columns:
['x_min', 'x_max', 'y_min', 'y_max', 'z_min', 'z_max']


In [7]:
centroid_voxel_columns = ['centroid_x_vox', 'centroid_y_vox', 'centroid_z_vox']

existing_centroid_column_sets = [
    ['centroid_x_vox', 'centroid_y_vox', 'centroid_z_vox'],
    ['x_centroid', 'y_centroid', 'z_centroid'],
    ['x_center', 'y_center', 'z_center'],
    ['x_mid', 'y_mid', 'z_mid'],
]

existing_centroid_columns = next(
    (columns for columns in existing_centroid_column_sets if set(columns).issubset(screened_root_with_annotation_df.columns)),
    None,
)

if existing_centroid_columns is not None:
    root_centroid_df = screened_root_with_annotation_df[['pt_root_id'] + existing_centroid_columns].copy()
    root_centroid_df = root_centroid_df.rename(columns=dict(zip(existing_centroid_columns, centroid_voxel_columns)))
    print(f'Using existing centroid columns: {existing_centroid_columns}')
else:
    print('No root-level centroid columns found; loading nuclei table to compute volume-weighted centroids.')
    nuclei_position_df = pd.read_parquet(
        nuclei_parquet_path,
        columns=['pt_root_id', 'pt_position', 'volume'],
    )
    nuclei_position_df = nuclei_position_df[nuclei_position_df['pt_root_id'] != 0].copy()
    nuclei_position_df['pt_root_id'] = pd.to_numeric(
        nuclei_position_df['pt_root_id'],
        errors='raise',
    ).astype('Int64')

    screened_root_id_set = set(screened_root_with_annotation_df['pt_root_id'].dropna().astype(int).tolist())
    nuclei_position_df = nuclei_position_df[nuclei_position_df['pt_root_id'].astype(int).isin(screened_root_id_set)].copy()

    position_xyz_df = pd.DataFrame(
        nuclei_position_df['pt_position'].tolist(),
        columns=centroid_voxel_columns,
        index=nuclei_position_df.index,
    )
    for column in centroid_voxel_columns:
        nuclei_position_df[column] = pd.to_numeric(position_xyz_df[column], errors='coerce')

    nuclei_position_df['centroid_weight'] = pd.to_numeric(
        nuclei_position_df['volume'],
        errors='coerce',
    ).fillna(0).clip(lower=0)

    for column in centroid_voxel_columns:
        nuclei_position_df[f'{column}_weighted'] = nuclei_position_df[column] * nuclei_position_df['centroid_weight']

    centroid_agg_df = (
        nuclei_position_df
        .groupby('pt_root_id')
        .agg(
            centroid_x_weighted_sum=('centroid_x_vox_weighted', 'sum'),
            centroid_y_weighted_sum=('centroid_y_vox_weighted', 'sum'),
            centroid_z_weighted_sum=('centroid_z_vox_weighted', 'sum'),
            centroid_weight_sum=('centroid_weight', 'sum'),
            centroid_x_mean=('centroid_x_vox', 'mean'),
            centroid_y_mean=('centroid_y_vox', 'mean'),
            centroid_z_mean=('centroid_z_vox', 'mean'),
            centroid_component_count=('pt_position', 'size'),
        )
        .reset_index()
    )

    for axis in ['x', 'y', 'z']:
        centroid_agg_df[f'centroid_{axis}_vox'] = np.where(
            centroid_agg_df['centroid_weight_sum'] > 0,
            centroid_agg_df[f'centroid_{axis}_weighted_sum'] / centroid_agg_df['centroid_weight_sum'],
            centroid_agg_df[f'centroid_{axis}_mean'],
        )

    root_centroid_df = centroid_agg_df[
        ['pt_root_id'] + centroid_voxel_columns + ['centroid_component_count', 'centroid_weight_sum']
    ].copy()

print('root centroid table shape:')
print(root_centroid_df.shape)
display(root_centroid_df.head())

No root-level centroid columns found; loading nuclei table to compute volume-weighted centroids.
root centroid table shape:
(13860, 6)


,pt_root_id,centroid_x_vox,centroid_y_vox,centroid_z_vox,centroid_component_count,centroid_weight_sum
0,648518346341364951,64112.00373,32240.001876,1032.541044,2,2.049131
1,648518346341389635,65824.00000,41008.000000,101.000000,1,0.612127
2,648518346341403465,65792.00000,40560.000000,113.000000,1,3.922837
3,648518346341404984,70000.00000,60512.000000,1947.000000,1,1.194394
4,648518346341407020,65760.00000,41200.000000,188.000000,1,1.593769


In [8]:
spatial_overview_df = screened_root_with_annotation_df.merge(
    root_centroid_df,
    on='pt_root_id',
    how='left',
    validate='one_to_one',
)

missing_centroid_mask = spatial_overview_df[centroid_voxel_columns].isna().any(axis=1)
missing_centroid_count = int(missing_centroid_mask.sum())

for axis in ['x', 'y', 'z']:
    spatial_overview_df[f'centroid_{axis}_um'] = (
        pd.to_numeric(spatial_overview_df[f'centroid_{axis}_vox'], errors='coerce') *
        voxel_resolution_nm[axis] / 1000
    )

spatial_plot_df = spatial_overview_df[~missing_centroid_mask].copy()

print(f'screened roots before centroid merge: {len(screened_root_with_annotation_df)}')
print(f'screened roots after centroid merge: {len(spatial_overview_df)}')
print(f'missing-centroid roots: {missing_centroid_count}')
print(f'roots included in 3D plot: {len(spatial_plot_df)}')
print('population counts in 3D plot:')
print(spatial_plot_df['population_label'].value_counts().to_dict())

if missing_centroid_count > 0:
    print('root IDs missing centroids:')
    print(spatial_overview_df.loc[missing_centroid_mask, 'pt_root_id'].dropna().astype(int).tolist())

screened roots before centroid merge: 13860
screened roots after centroid merge: 13860
missing-centroid roots: 0
roots included in 3D plot: 13860
population counts in 3D plot:
{'unlabeled': 11799, 'annotated_neuron': 2061}


## Plot Root Centroids

This is the simpler/default Plotly view. Each screened root is represented by a 3D centroid marker; color distinguishes `annotated_neuron` from `unlabeled`, and marker size is derived from transformed `nucleus_volume_sum`.

In [9]:
# ------------------------------------------------------------
# USER SETTINGS: CENTROID VIEW
# ------------------------------------------------------------
population_plot_order = ['annotated_neuron', 'unlabeled']
population_plot_colors = {
    'annotated_neuron': '#1f77b4',
    'unlabeled': '#ff7f0e',
}
population_marker_opacity = {
    'annotated_neuron': 0.75, #0.75,
    'unlabeled': 0.75, #0.35,
}

nucleus_volume_size_transform = 'log'
nucleus_volume_marker_size_min = 0.5 #2.0
nucleus_volume_marker_size_max = 3 # 12.0

def transform_nucleus_volume_for_marker_size(values, method):
    values = pd.to_numeric(values, errors='coerce').astype(float)
    method = method.lower() if isinstance(method, str) else method

    if method is None:
        transformed_values = values
    elif method in ['log', 'log10']:
        transformed_values = np.log10(values.where(values > 0))
    elif method == 'log1p':
        transformed_values = np.log1p(values.clip(lower=0))
    elif method == 'sqrt':
        transformed_values = np.sqrt(values.clip(lower=0))
    elif method in ['cuberoot', 'cube_root']:
        transformed_values = np.cbrt(values.clip(lower=0))
    elif method == 'rank':
        transformed_values = values.rank(pct=True)
    else:
        raise ValueError(
            "nucleus_volume_size_transform must be one of "
            "None, 'log', 'log10', 'log1p', 'sqrt', 'cuberoot', or 'rank'."
        )

    return transformed_values.replace([np.inf, -np.inf], np.nan)

nucleus_volume_size_values = transform_nucleus_volume_for_marker_size(
    spatial_plot_df['nucleus_volume_sum'],
    nucleus_volume_size_transform,
)
nucleus_volume_size_min = nucleus_volume_size_values.min(skipna=True)
nucleus_volume_size_max = nucleus_volume_size_values.max(skipna=True)

if pd.isna(nucleus_volume_size_min) or nucleus_volume_size_min == nucleus_volume_size_max:
    spatial_plot_df['nucleus_volume_marker_size'] = (
        nucleus_volume_marker_size_min + nucleus_volume_marker_size_max
    ) / 2
else:
    spatial_plot_df['nucleus_volume_marker_size'] = nucleus_volume_marker_size_min + (
        (nucleus_volume_size_values - nucleus_volume_size_min) /
        (nucleus_volume_size_max - nucleus_volume_size_min)
    ) * (nucleus_volume_marker_size_max - nucleus_volume_marker_size_min)
    spatial_plot_df['nucleus_volume_marker_size'] = spatial_plot_df['nucleus_volume_marker_size'].fillna(
        nucleus_volume_marker_size_min
    )

fig = go.Figure()
for population_label in population_plot_order:
    population_df = spatial_plot_df[spatial_plot_df['population_label'] == population_label].copy()
    if population_df.empty:
        continue

    customdata_columns = ['pt_root_id', 'population_label']
    for optional_column in ['nuclei_rows', 'nucleus_volume_sum']:
        if optional_column in population_df.columns:
            customdata_columns.append(optional_column)

    fig.add_trace(
        go.Scatter3d(
            x=population_df['centroid_x_um'],
            y=population_df['centroid_y_um'],
            z=population_df['centroid_z_um'],
            mode='markers',
            name=population_label,
            customdata=population_df[customdata_columns],
            marker={
                'size': population_df['nucleus_volume_marker_size'],
                'opacity': population_marker_opacity.get(population_label, 0.45),
                'color': population_plot_colors.get(population_label, '#666666'),
            },
            hovertemplate=(
                'root_id=%{customdata[0]}<br>'
                'population=%{customdata[1]}<br>'
                'nuclei_rows=%{customdata[2]}<br>'
                'nucleus_volume_sum=%{customdata[3]:.3f}<br>'
                'x=%{x:.2f} um<br>'
                'y=%{y:.2f} um<br>'
                'z=%{z:.2f} um'
                '<extra></extra>'
            ),
        )
    )

fig.update_layout(
    title=f'Screened root population centroid overview; marker size = nucleus_volume_sum ({nucleus_volume_size_transform})',
    scene={
        'xaxis_title': 'x centroid (um; 18 nm voxels)',
        'yaxis_title': 'y centroid (um; 18 nm voxels)',
        'zaxis_title': 'z centroid (um; 45 nm sections)',
        'aspectmode': 'data',
    },
    legend={
        'title': {'text': 'Population label'},
        'itemsizing': 'constant',
    },
    margin={'l': 0, 'r': 0, 'b': 0, 't': 45},
    height=800,
)

if plotly_renderer:
    fig.show(renderer=plotly_renderer)
else:
    fig.show()


## Add Local Mesh Overlays

Mesh overlays are optional. `mesh_root_ids` determines which roots are requested, only locally available meshes are added, and the remaining centroid population stays visible for spatial context.


In [10]:
# ------------------------------------------------------------
# USER SETTINGS: MESH SELECTION
# ------------------------------------------------------------
mesh_root_ids = [648518346432510775, 648518346436805246, 648518346440739174, 648518346441158615, 648518346441998053, 648518346444264392, 648518346444314087, 648518346445010973, 648518346448038906, 648518346449469764]

decimated_mesh_paths, decimated_mesh_manifest_df = refresh_decimated_mesh_manifest()

available_decimated_mesh_root_ids = set(
    decimated_mesh_manifest_df['pt_root_id']
    .dropna()
    .astype(int)
    .tolist()
)

present_mesh_root_ids = [
    int(root_id) for root_id in mesh_root_ids
    if int(root_id) in available_decimated_mesh_root_ids
]
missing_mesh_root_ids = [
    int(root_id) for root_id in mesh_root_ids
    if int(root_id) not in available_decimated_mesh_root_ids
]

print(f'mesh root IDs requested: {len(mesh_root_ids)}')
print(f'decimated meshes present: {len(present_mesh_root_ids)}')
print(present_mesh_root_ids)
print(f'decimated meshes missing: {len(missing_mesh_root_ids)}')
print(missing_mesh_root_ids)


mesh root IDs requested: 10
decimated meshes present: 10
[648518346432510775, 648518346436805246, 648518346440739174, 648518346441158615, 648518346441998053, 648518346444264392, 648518346444314087, 648518346445010973, 648518346448038906, 648518346449469764]
decimated meshes missing: 0
[]


In [11]:
def read_ply_header_summary(mesh_path):
    vertex_count = None
    face_count = None
    ply_format = None

    with mesh_path.open('rb') as file:
        for raw_line in file:
            line = raw_line.decode('ascii', errors='ignore').strip()
            if line.startswith('format '):
                ply_format = line.split()[1]
            elif line.startswith('element vertex '):
                vertex_count = int(line.split()[-1])
            elif line.startswith('element face '):
                face_count = int(line.split()[-1])
            elif line == 'end_header':
                break

    return {
        'mesh_path': mesh_path,
        'mesh_filename': mesh_path.name,
        'ply_format': ply_format,
        'vertex_count': vertex_count,
        'face_count': face_count,
        'file_size_mb': mesh_path.stat().st_size / 1_000_000,
    }

decimated_mesh_paths, decimated_mesh_manifest_df = refresh_decimated_mesh_manifest()
mesh_manifest_by_root_id = (
    decimated_mesh_manifest_df
    .dropna(subset=['pt_root_id'])
    .assign(pt_root_id=lambda df: df['pt_root_id'].astype(int))
    .set_index('pt_root_id')
)

mesh_summary_rows = []
for root_id in mesh_root_ids:
    root_id = int(root_id)
    if root_id not in mesh_manifest_by_root_id.index:
        mesh_summary_rows.append({
            'pt_root_id': root_id,
            'mesh_present': False,
            'mesh_path': None,
            'mesh_filename': None,
            'ply_format': None,
            'vertex_count': None,
            'face_count': None,
            'file_size_mb': None,
        })
        continue

    mesh_record = mesh_manifest_by_root_id.loc[root_id]
    if isinstance(mesh_record, pd.DataFrame):
        mesh_record = mesh_record.iloc[0]

    mesh_summary = read_ply_header_summary(mesh_record['mesh_path'])
    mesh_summary_rows.append({
        'pt_root_id': root_id,
        'mesh_present': True,
        **mesh_summary,
    })

mesh_size_summary_df = pd.DataFrame(mesh_summary_rows)

print(f'meshes summarized: {int(mesh_size_summary_df["mesh_present"].sum())}')
print(f'meshes missing: {int((~mesh_size_summary_df["mesh_present"]).sum())}')
mesh_size_summary_display_df = mesh_size_summary_df.copy()
if 'mesh_path' in mesh_size_summary_display_df.columns:
    mesh_size_summary_display_df['mesh_path'] = mesh_size_summary_display_df['mesh_path'].map(
        lambda path: format_path(path, project_root, show_full_path) if path is not None else None
    )
display(mesh_size_summary_display_df)


meshes summarized: 10
meshes missing: 0


,pt_root_id,mesh_present,mesh_path,mesh_filename,ply_format,vertex_count,face_count,file_size_mb
0,648518346432510775,True,data\meshes\dec\mesh_648518346432510775_mat195...,mesh_648518346432510775_mat195_dec95.ply,binary_little_endian,49285,98435,2.462806
1,648518346436805246,True,data\meshes\dec\mesh_648518346436805246_mat195...,mesh_648518346436805246_mat195_dec95.ply,binary_little_endian,31705,63015,1.580426
2,648518346440739174,True,data\meshes\dec\mesh_648518346440739174_mat195...,mesh_648518346440739174_mat195_dec95.ply,binary_little_endian,40691,79499,2.010382
3,648518346441158615,True,data\meshes\dec\mesh_648518346441158615_mat195...,mesh_648518346441158615_mat195_dec95.ply,binary_little_endian,50633,101222,2.531390
4,648518346441998053,True,data\meshes\dec\mesh_648518346441998053_mat195...,mesh_648518346441998053_mat195_dec95.ply,binary_little_endian,85537,170717,4.272521
5,648518346444264392,True,data\meshes\dec\mesh_648518346444264392_mat195...,mesh_648518346444264392_mat195_dec95.ply,binary_little_endian,42847,83807,2.118130
6,648518346444314087,True,data\meshes\dec\mesh_648518346444314087_mat195...,mesh_648518346444314087_mat195_dec95.ply,binary_little_endian,70663,141304,3.533176
7,648518346445010973,True,data\meshes\dec\mesh_648518346445010973_mat195...,mesh_648518346445010973_mat195_dec95.ply,binary_little_endian,64398,125193,3.173373
8,648518346448038906,True,data\meshes\dec\mesh_648518346448038906_mat195...,mesh_648518346448038906_mat195_dec95.ply,binary_little_endian,50004,97390,2.466477
9,648518346449469764,True,data\meshes\dec\mesh_648518346449469764_mat195...,mesh_648518346449469764_mat195_dec95.ply,binary_little_endian,76981,149517,3.791577


In [12]:
import pyvista as pv

# ------------------------------------------------------------
# USER SETTINGS: MESH OVERLAY
# ------------------------------------------------------------
mesh_face_sample_fraction = 0.5
mesh_face_sample_random_state = 195
mesh_vertex_units = 'nm'
mesh_overlay_opacity = 1
mesh_overlay_colors = [
    '#4c78a8',
    '#f58518',
    '#54a24b',
    '#e45756',
    '#72b7b2',
    '#b279a2',
    '#ff9da6',
    '#9d755d',
    '#bab0ac',
    '#8cd17d',
]

def mesh_vertices_to_um(vertices, units):
    if units == 'nm':
        return vertices / 1000
    if units == 'um':
        return vertices
    if units == 'vox':
        return vertices * np.array([
            voxel_resolution_nm['x'],
            voxel_resolution_nm['y'],
            voxel_resolution_nm['z'],
        ]) / 1000
    raise ValueError("mesh_vertex_units must be one of 'nm', 'um', or 'vox'.")

def sample_mesh_faces_for_plotly(mesh_path, sample_fraction, random_generator):
    mesh = pv.read(mesh_path)
    vertices = np.asarray(mesh.points, dtype=float)
    faces_with_sizes = np.asarray(mesh.faces)

    if len(vertices) == 0 or len(faces_with_sizes) == 0:
        raise ValueError(f'Empty mesh: {mesh_path}')

    faces_with_sizes = faces_with_sizes.reshape(-1, 4)
    if not np.all(faces_with_sizes[:, 0] == 3):
        raise ValueError(f'Only triangular PLY faces are supported: {mesh_path}')

    faces = faces_with_sizes[:, 1:4].astype(int)
    sampled_face_count = max(1, int(np.ceil(len(faces) * sample_fraction)))
    sampled_face_count = min(sampled_face_count, len(faces))
    sampled_face_indices = random_generator.choice(
        len(faces),
        size=sampled_face_count,
        replace=False,
    )
    sampled_faces = faces[sampled_face_indices]

    used_vertex_indices, compact_faces = np.unique(sampled_faces, return_inverse=True)
    compact_vertices = vertices[used_vertex_indices]
    compact_faces = compact_faces.reshape(-1, 3)

    return mesh_vertices_to_um(compact_vertices, mesh_vertex_units), compact_faces, len(faces)

fig_with_mesh = go.Figure(fig)
mesh_overlay_summary_rows = []
mesh_random_generator = np.random.default_rng(mesh_face_sample_random_state)

for mesh_index, row in mesh_size_summary_df.loc[mesh_size_summary_df['mesh_present']].iterrows():
    root_id = int(row['pt_root_id'])
    mesh_vertices_um, mesh_faces, original_face_count = sample_mesh_faces_for_plotly(
        row['mesh_path'],
        mesh_face_sample_fraction,
        mesh_random_generator,
    )
    mesh_color = mesh_overlay_colors[mesh_index % len(mesh_overlay_colors)]

    fig_with_mesh.add_trace(
        go.Mesh3d(
            x=mesh_vertices_um[:, 0],
            y=mesh_vertices_um[:, 1],
            z=mesh_vertices_um[:, 2],
            i=mesh_faces[:, 0],
            j=mesh_faces[:, 1],
            k=mesh_faces[:, 2],
            name=f'mesh {root_id}',
            color=mesh_color,
            opacity=mesh_overlay_opacity,
            flatshading=True,
            hoverinfo='skip',
            showscale=False,
        )
    )

    mesh_overlay_summary_rows.append({
        'pt_root_id': root_id,
        'original_vertex_count': int(row['vertex_count']),
        'original_face_count': int(original_face_count),
        'sampled_vertex_count': len(mesh_vertices_um),
        'sampled_face_count': len(mesh_faces),
        'sample_fraction': mesh_face_sample_fraction,
    })

mesh_overlay_summary_df = pd.DataFrame(mesh_overlay_summary_rows)

print(f'mesh traces added: {len(mesh_overlay_summary_df)}')
print(f'total sampled vertices: {mesh_overlay_summary_df["sampled_vertex_count"].sum()}')
print(f'total sampled faces: {mesh_overlay_summary_df["sampled_face_count"].sum()}')
display(mesh_overlay_summary_df)

if plotly_renderer:
    fig_with_mesh.show(renderer=plotly_renderer)
else:
    fig_with_mesh.show()


mesh traces added: 10
total sampled vertices: 544156
total sampled faces: 555053


,pt_root_id,original_vertex_count,original_face_count,sampled_vertex_count,sampled_face_count,sample_fraction
0,648518346432510775,49285,98435,47906,49218,0.5
1,648518346436805246,31705,63015,30788,31508,0.5
2,648518346440739174,40691,79499,39221,39750,0.5
3,648518346441158615,50633,101222,49139,50611,0.5
4,648518346441998053,85537,170717,82921,85359,0.5
5,648518346444264392,42847,83807,41328,41904,0.5
6,648518346444314087,70663,141304,68576,70652,0.5
7,648518346445010973,64398,125193,61946,62597,0.5
8,648518346448038906,50004,97390,48301,48695,0.5
9,648518346449469764,76981,149517,74030,74759,0.5
